# 05 — Detección online de cambios

> Esta notebook evalúa si un detector bayesiano online añade información al diagnóstico rolling de 21 sesiones sobre la relación entre el error de atribución de XLE y el mercado. Las alertas son exploratorias: una ruptura del diagnóstico no identifica por sí sola un cambio verdadero de beta.

## <b>Modelos congelados y datos</b>

Se reproducen rolling OLS de 252 sesiones y Kalman con $Q/R$ fijos de la notebook 02. Para cada fecha $t$, ambos atribuyen el retorno con coeficientes disponibles hasta $t-1$ y con factores realizados en $t$. La evaluación abarca 2022–2026; la calibración termina en 2021.

In [1]:
from datetime import date
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

TARGETS = ('QQQ', 'XLE', 'XLF')
SYMBOLS = ('SPY', 'QQQ', 'IWM', 'IWD', 'IWF', 'XLE', 'XLF')
FACTORS = ('market', 'size', 'growth_value')
RESEARCH_START, RESEARCH_END = date(2018, 1, 2), date(2026, 11, 17)
CALIBRATION_END, EVALUATION_START = date(2021, 12, 31), date(2022, 1, 1)
ROLLING_WINDOW, DIAGNOSTIC_WINDOW = 252, 21

def project_root(start):
    for parent in (start.resolve(), *start.resolve().parents):
        direct = parent / 'Momentum_systematic_strategies'
        if (direct / 'data' / 'gold' / 'daily_series' / 'etf').is_dir():
            return direct
        if (parent / 'data' / 'gold' / 'daily_series' / 'etf').is_dir():
            return parent
    raise FileNotFoundError('No se encontró data/gold/daily_series/etf.')

GOLD = project_root(Path.cwd()) / 'data' / 'gold' / 'daily_series' / 'etf'
frames = {symbol: pl.read_parquet(GOLD / f'{symbol}.parquet').with_columns(pl.col('session_date').cast(pl.Date)).filter(
    pl.col('session_date').is_between(RESEARCH_START, RESEARCH_END, closed='both')).sort('session_date') for symbol in SYMBOLS}
prices = frames['SPY'].select(pl.col('session_date').alias('date'), pl.col('adjusted_close').alias('SPY'))
for symbol in SYMBOLS[1:]:
    prices = prices.join(frames[symbol].select(pl.col('session_date').alias('date'), pl.col('adjusted_close').alias(symbol)), on='date', how='inner')
returns = prices.sort('date').with_columns([pl.col(symbol).log().diff().alias(f'{symbol}_ret') for symbol in SYMBOLS]).drop_nulls()
model_panel = returns.select('date', *[f'{asset}_ret' for asset in TARGETS], 'SPY_ret', 'IWM_ret', 'IWF_ret', 'IWD_ret').with_columns(
    pl.col('SPY_ret').alias('market'), (pl.col('IWM_ret') - pl.col('SPY_ret')).alias('size'),
    (pl.col('IWF_ret') - pl.col('IWD_ret')).alias('growth_value')).select('date', *[f'{asset}_ret' for asset in TARGETS], *FACTORS).drop_nulls()
dates, factor_values = model_panel.get_column('date').to_list(), model_panel.select(*FACTORS).to_numpy()
evaluation_index = next(index for index, value in enumerate(dates) if value >= EVALUATION_START)
assert dates[evaluation_index - 1] <= CALIBRATION_END


In [2]:
ols_rows, calibration = [], {}
for asset in TARGETS:
    actual = model_panel.get_column(f'{asset}_ret').to_numpy()
    history, residual_history = [], []
    for index in range(ROLLING_WINDOW, len(actual)):
        design = np.column_stack((np.ones(ROLLING_WINDOW), factor_values[index - ROLLING_WINDOW:index]))
        beta = np.linalg.lstsq(design, actual[index - ROLLING_WINDOW:index], rcond=None)[0]
        predicted = np.r_[1.0, factor_values[index]] @ beta
        residual = actual[index] - predicted
        ols_rows.append({'date': dates[index], 'beta_asof_date': dates[index - 1], 'asset': asset,
                         'alpha': beta[0], 'beta_market': beta[1], 'beta_size': beta[2], 'beta_growth_value': beta[3],
                         'actual_return': actual[index], 'predicted_return': predicted, 'residual_return': residual})
        if dates[index] <= CALIBRATION_END:
            history.append(beta)
            residual_history.append(residual)
    history = np.asarray(history)
    r = float(np.var(residual_history, ddof=1))
    last_design = np.column_stack((np.ones(ROLLING_WINDOW), factor_values[evaluation_index - ROLLING_WINDOW:evaluation_index]))
    calibration[asset] = {'Q': np.diag(np.var(np.diff(history, axis=0), axis=0, ddof=1)), 'R': r,
                          'theta': history[-1], 'P': r * np.linalg.pinv(last_design.T @ last_design)}
ols = pl.DataFrame(ols_rows)

def run_kalman(asset):
    actual = model_panel.get_column(f'{asset}_ret').to_numpy()
    theta, covariance = calibration[asset]['theta'].copy(), calibration[asset]['P'].copy()
    rows = []
    for index in range(evaluation_index, len(actual)):
        x = np.r_[1.0, factor_values[index]]
        theta_prior, covariance_prior = theta, covariance + calibration[asset]['Q']
        predicted, error = x @ theta_prior, actual[index] - x @ theta_prior
        gain = covariance_prior @ x / float(x @ covariance_prior @ x + calibration[asset]['R'])
        theta = theta_prior + gain * error
        update = np.eye(4) - np.outer(gain, x)
        covariance = update @ covariance_prior @ update.T + np.outer(gain, gain) * calibration[asset]['R']
        rows.append({'date': dates[index], 'asset': asset, 'actual_return': actual[index],
                     'predicted_return': predicted, 'residual_return': error})
    return pl.DataFrame(rows)

kalman = pl.concat([run_kalman(asset) for asset in TARGETS])
ols_evaluation = ols.filter(pl.col('date') >= EVALUATION_START)
reference_r2 = {'QQQ': (0.9686, 0.9708), 'XLE': (0.3164, 0.3351), 'XLF': (0.7725, 0.7792)}
reproduction_rows = []
for asset in TARGETS:
    values = []
    for frame in (ols_evaluation, kalman):
        sample = frame.filter(pl.col('asset') == asset)
        actual, error = sample.get_column('actual_return').to_numpy(), sample.get_column('residual_return').to_numpy()
        values.append(1 - (error ** 2).sum() / ((actual - actual.mean()) ** 2).sum())
    reproduction_rows.append({'asset': asset, 'ols_r2': values[0], 'kalman_r2': values[1],
                             'reference_ols_r2': reference_r2[asset][0], 'reference_kalman_r2': reference_r2[asset][1]})
if any(abs(row['ols_r2'] - row['reference_ols_r2']) > .0001 or abs(row['kalman_r2'] - row['reference_kalman_r2']) > .0001 for row in reproduction_rows):
    raise AssertionError('Los modelos congelados no reproducen la notebook 02.')
reproduction = pl.DataFrame(reproduction_rows).with_columns(pl.all().exclude('asset').round(4))
common_observations = ols_evaluation.select('date', 'asset').join(kalman.select('date', 'asset'), on=['date', 'asset']).height
reproduction, common_observations


(shape: (3, 5)
 ┌───────┬────────┬───────────┬──────────────────┬─────────────────────┐
 │ asset ┆ ols_r2 ┆ kalman_r2 ┆ reference_ols_r2 ┆ reference_kalman_r2 │
 │ ---   ┆ ---    ┆ ---       ┆ ---              ┆ ---                 │
 │ str   ┆ f64    ┆ f64       ┆ f64              ┆ f64                 │
 ╞═══════╪════════╪═══════════╪══════════════════╪═════════════════════╡
 │ QQQ   ┆ 0.9686 ┆ 0.9708    ┆ 0.9686           ┆ 0.9708              │
 │ XLE   ┆ 0.3164 ┆ 0.3351    ┆ 0.3164           ┆ 0.3351              │
 │ XLF   ┆ 0.7725 ┆ 0.7792    ┆ 0.7725           ┆ 0.7792              │
 └───────┴────────┴───────────┴──────────────────┴─────────────────────┘,
 3672)

## <b>Diagnóstico rolling de referencia</b>

> Se conserva el diagnóstico de la notebook 04: $m_{MKT,t}$ es la media móvil de 21 sesiones de $MKT_t e_t$. El umbral es el percentil 97.5 de su magnitud antes de 2022. Las alertas consecutivas se solapan y no son observaciones independientes.

In [3]:
xle_ols_all = ols.filter(pl.col('asset') == 'XLE').sort('date')
xle_factors = xle_ols_all.join(model_panel.select('date', *FACTORS), on='date', how='inner').sort('date')
all_dates = xle_factors.get_column('date').to_list()
calibration_mask = np.asarray([value <= CALIBRATION_END for value in all_dates])
evaluation_mask = np.asarray([value >= EVALUATION_START for value in all_dates])
ols_error = xle_factors.get_column('residual_return').to_numpy()
mkt_return = xle_factors.get_column('market').to_numpy()
diagnostic_daily = mkt_return * ols_error
rolling_mkt = np.full(len(ols_error), np.nan)
for index in range(DIAGNOSTIC_WINDOW - 1, len(ols_error)):
    rolling_mkt[index] = diagnostic_daily[index - DIAGNOSTIC_WINDOW + 1:index + 1].mean()
rolling_threshold = np.quantile(np.abs(rolling_mkt[calibration_mask & np.isfinite(rolling_mkt)]), .975)
rolling_alert = np.abs(rolling_mkt) > rolling_threshold

def consecutive_blocks(values):
    blocks, start = [], None
    for index, value in enumerate(values):
        if value and start is None:
            start = index
        if start is not None and (not value or index == len(values) - 1):
            stop = index if not value else index + 1
            blocks.append((start, stop))
            start = None
    return blocks

rolling_eval = rolling_alert[evaluation_mask]
rolling_blocks = consecutive_blocks(rolling_eval)
pl.DataFrame([{'rolling_threshold': rolling_threshold, 'alert_sessions': int(rolling_eval.sum()),
               'episodes': len(rolling_blocks), 'median_duration': np.median([stop - start for start, stop in rolling_blocks]),
               'max_duration': max(stop - start for start, stop in rolling_blocks)}]).with_columns(pl.all().round(6))


rolling_threshold,alert_sessions,episodes,median_duration,max_duration
f64,i64,i64,f64,i64
0.000154,72,4,18.0,21


## <b>BOCPD sobre el producto firmado</b>

> El detector observa $y_t=MKT_t e_t$. Dentro de cada segmento asume observaciones normales independientes con varianza conocida $s^2$ y media desconocida. La media tiene un prior normal con fuerza inicial 21. La escala $s^2$, la media previa y el piso de cálculo se calibran antes de 2022; el hazard fijo $H=1/63$ implica una duración previa esperada de 63 sesiones y se define antes de revisar alertas posteriores.

El run length es el número de observaciones desde el último cambio hipotético. Después de observar $y_t$, se registra la probabilidad posterior de cambio. La alerta usa la probabilidad de que el run length sea menor a 21 y un umbral del percentil 97.5 pre-2022; las primeras 21 observaciones se excluyen sólo de ese percentil porque su run length corto es mecánico durante la inicialización. La recursión sólo utiliza datos observados hasta cada fecha.

In [4]:
calibration_values = diagnostic_daily[calibration_mask]
prior_mean = float(calibration_values.mean())
observation_variance = float(np.var(calibration_values, ddof=1))
prior_strength, hazard = 21.0, 1 / 63

def run_bocpd(values, mean0, variance, kappa0, hazard_rate):
    probabilities = np.array([1.0])
    means, kappas = np.array([mean0]), np.array([kappa0])
    change_probability, short_run_probability, map_run = [], [], []
    for value in values:
        predictive_variance = variance * (1 + 1 / kappas)
        density = np.exp(-.5 * (value - means) ** 2 / predictive_variance) / np.sqrt(2 * np.pi * predictive_variance)
        reset_variance = variance * (1 + 1 / kappa0)
        reset_density = np.exp(-.5 * (value - mean0) ** 2 / reset_variance) / np.sqrt(2 * np.pi * reset_variance)
        joint = np.r_[hazard_rate * reset_density, probabilities * (1 - hazard_rate) * density]
        probabilities = joint / joint.sum()
        means = np.r_[(kappa0 * mean0 + value) / (kappa0 + 1), (kappas * means + value) / (kappas + 1)]
        kappas = np.r_[kappa0 + 1, kappas + 1]
        change_probability.append(probabilities[0])
        short_run_probability.append(probabilities[:min(21, len(probabilities))].sum())
        map_run.append(int(np.argmax(probabilities)))
    return np.asarray(change_probability), np.asarray(short_run_probability), np.asarray(map_run)

bocpd_change, bocpd_short_run, bocpd_map_run = run_bocpd(diagnostic_daily, prior_mean, observation_variance, prior_strength, hazard)
warmup_mask = np.arange(len(all_dates)) >= DIAGNOSTIC_WINDOW
bocpd_threshold = np.quantile(bocpd_short_run[calibration_mask & warmup_mask], .975)
bocpd_alert = bocpd_short_run > bocpd_threshold
assert np.isfinite(bocpd_short_run).all() and np.isfinite(bocpd_change).all()
assert len(bocpd_short_run[evaluation_mask]) == 1224

bocpd_eval = bocpd_alert[evaluation_mask]
bocpd_blocks = consecutive_blocks(bocpd_eval)
comparison = pl.DataFrame([
    {'detector': 'Rolling MKT 21d', 'alert_sessions': int(rolling_eval.sum()), 'episodes': len(rolling_blocks),
     'median_duration': float(np.median([stop - start for start, stop in rolling_blocks])), 'max_duration': max(stop - start for start, stop in rolling_blocks)},
    {'detector': 'BOCPD short run <21d', 'alert_sessions': int(bocpd_eval.sum()), 'episodes': len(bocpd_blocks),
     'median_duration': float(np.median([stop - start for start, stop in bocpd_blocks])) if bocpd_blocks else 0.0,
     'max_duration': max([stop - start for start, stop in bocpd_blocks], default=0)},
]).with_columns(pl.all().exclude('detector').round(2))
parameters = pl.DataFrame([{'prior_mean': prior_mean, 'observation_std': np.sqrt(observation_variance),
                        'prior_strength': prior_strength, 'hazard': hazard, 'expected_run_length': 1 / hazard,
                        'bocpd_alert_threshold': bocpd_threshold}]).with_columns(pl.all().round(6))
parameters, comparison


(shape: (1, 6)
 ┌────────────┬─────────────────┬────────────────┬──────────┬───────────────────┬───────────────────┐
 │ prior_mean ┆ observation_std ┆ prior_strength ┆ hazard   ┆ expected_run_leng ┆ bocpd_alert_thres │
 │ ---        ┆ ---             ┆ ---            ┆ ---      ┆ th                ┆ hold              │
 │ f64        ┆ f64             ┆ f64            ┆ f64      ┆ ---               ┆ ---               │
 │            ┆                 ┆                ┆          ┆ f64               ┆ f64               │
 ╞════════════╪═════════════════╪════════════════╪══════════╪═══════════════════╪═══════════════════╡
 │ -0.000002  ┆ 0.000432        ┆ 21.0           ┆ 0.015873 ┆ 63.0              ┆ 0.988871          │
 └────────────┴─────────────────┴────────────────┴──────────┴───────────────────┴───────────────────┘,
 shape: (2, 5)
 ┌──────────────────────┬────────────────┬──────────┬─────────────────┬──────────────┐
 │ detector             ┆ alert_sessions ┆ episodes ┆ median_durat

## <b>Comparación alrededor de los episodios históricos</b>

> Los tres casos se conservan como estudios de caso retrospectivos: 2022-03-07, 2025-04-04 y 2026-04-08. Fueron priorizados con información de los 21 días posteriores en la notebook 04; no son fechas que un detector necesariamente habría elegido en tiempo real.

In [5]:
episode_dates = [date(2022, 3, 7), date(2025, 4, 4), date(2026, 4, 8)]
xle_kalman = kalman.filter(pl.col('asset') == 'XLE').sort('date')
xle_ols_evaluation = xle_ols_all.filter(pl.col('date') >= EVALUATION_START).sort('date')
evaluation_dates = xle_ols_evaluation.get_column('date').to_list()
case_rows = []
for episode, episode_date in enumerate(episode_dates, start=1):
    all_index = all_dates.index(episode_date)
    evaluation_position = evaluation_dates.index(episode_date)
    future = slice(evaluation_position + 1, evaluation_position + 22)
    ols_future = xle_ols_evaluation.get_column('residual_return').to_numpy()[future]
    kalman_future = xle_kalman.get_column('residual_return').to_numpy()[future]
    case_rows.append({'episode': episode, 'date': episode_date, 'rolling_score': abs(rolling_mkt[all_index]) / rolling_threshold,
                      'bocpd_short_run_prob': bocpd_short_run[all_index], 'bocpd_change_prob': bocpd_change[all_index],
                      'map_run_length': bocpd_map_run[all_index], 'bocpd_alerts_next21': int(bocpd_alert[all_index:all_index + 21].sum()),
                      'bocpd_max_prob_next21': bocpd_short_run[all_index:all_index + 21].max(),
                      'ols_mae_next21_pct': np.abs(ols_future).mean() * 100, 'kalman_mae_next21_pct': np.abs(kalman_future).mean() * 100,
                      'ols_rmse_next21_pct': np.sqrt(np.mean(ols_future ** 2)) * 100, 'kalman_rmse_next21_pct': np.sqrt(np.mean(kalman_future ** 2)) * 100})
case_studies = pl.DataFrame(case_rows).with_columns(pl.all().exclude('episode', 'date', 'map_run_length', 'bocpd_alerts_next21').round(3))
case_studies


episode,date,rolling_score,bocpd_short_run_prob,bocpd_change_prob,map_run_length,bocpd_alerts_next21,bocpd_max_prob_next21,ols_mae_next21_pct,kalman_mae_next21_pct,ols_rmse_next21_pct,kalman_rmse_next21_pct
i64,date,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64
1,2022-03-07,1.09,0.286,0.014,18,0,0.306,1.74,1.699,2.107,2.068
2,2025-04-04,1.322,0.461,0.012,1,0,0.461,0.953,0.993,1.187,1.258
3,2026-04-08,1.292,0.282,0.013,8,0,0.282,1.195,1.196,1.614,1.536


## <b>Visualización</b>

La primera figura confronta la probabilidad de run corto de BOCPD con las fechas de alerta rolling. La segunda muestra el comportamiento de los errores de atribución en los tres casos, sin interpretar las fechas de alerta como la fecha real de una ruptura.

In [6]:
figure = go.Figure()
figure.add_scatter(x=all_dates, y=bocpd_short_run, mode='lines', name='BOCPD: P(run < 21)')
figure.add_hline(y=bocpd_threshold, line_dash='dash', line_color='firebrick', annotation_text='umbral BOCPD')
rolling_dates = [all_dates[index] for index in np.flatnonzero(rolling_alert & evaluation_mask)]
figure.add_scatter(x=rolling_dates, y=[bocpd_short_run[all_dates.index(value)] for value in rolling_dates], mode='markers',
                   marker={'size': 5, 'color': 'gray'}, name='alerta rolling MKT')
for value in episode_dates:
    figure.add_vline(x=value, line_dash='dot', line_color='black')
figure.update_layout(title='XLE — BOCPD y alertas rolling', xaxis_title='fecha', yaxis_title='probabilidad posterior', template='plotly_white')
figure.show()


In [7]:
figure = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=[str(value) for value in episode_dates])
ols_values = xle_ols_evaluation.get_column('residual_return').to_numpy() * 100
kalman_values = xle_kalman.get_column('residual_return').to_numpy() * 100
for column, episode_date in enumerate(episode_dates, start=1):
    position = evaluation_dates.index(episode_date)
    window = slice(max(0, position - 10), min(len(evaluation_dates), position + 22))
    relative_day = np.arange(window.start - position, window.stop - position)
    figure.add_scatter(x=relative_day, y=ols_values[window], mode='lines', name='Rolling OLS', legendgroup='ols', showlegend=column == 1, row=1, col=column)
    figure.add_scatter(x=relative_day, y=kalman_values[window], mode='lines', name='Kalman fijo', legendgroup='kalman', showlegend=column == 1, row=1, col=column)
    figure.add_vline(x=0, line_dash='dot', line_color='black', row=1, col=column)
figure.update_layout(title='XLE — errores alrededor de los estudios de caso', template='plotly_white', yaxis_title='error de atribución (%)')
figure.update_xaxes(title_text='sesiones desde alerta')
figure.show()


## <b>Conclusión</b>

- La alerta rolling MKT identifica 72 sesiones agrupadas en 4 episodios; con la especificación BOCPD congelada no se supera el umbral histórico en 2022–2026.
- En los tres estudios de caso, la probabilidad de run corto permanece por debajo del umbral, incluso cuando la alerta rolling persiste. En esta muestra, BOCPD no añade un episodio operativo adicional.
- Esta diferencia no demuestra que las alertas rolling sean cambios estructurales: las ventanas móviles comparten observaciones y BOCPD puede ser conservador frente a desplazamientos de media breves.
- Los errores posteriores de OLS y Kalman describen respuesta de atribución, no una verdad de referencia sobre cambios estructurales.
- Con estos parámetros no hay evidencia suficiente para usar BOCPD como capa adicional de decisión económica; sería necesario validarlo prospectivamente antes de aumentar complejidad.